# Automated Blood Cell Classification Using Advanced Deep Learning Techniques

Blood cells are specialised cells in the bloodstream that perform vital functions which includes transporting oxygen, fighting infections and facilitating blood clotting. They are primarily classified into three main types: red blood cells, white blood cells and platelets

The classes of white blood cells include eosinophils, lymphocytes, monocytes and neutrophils, each plays unique roles in the immune response.
Eosinophils combat parasitic infections, lymphocytes are involved in adaptive immunity, monocytes differentiate into macrophages for phagocytosis and neutrophils are the first responders to infection.

https://www.kaggle.com/datasets/paultimothymooney/blood-cells

In [7]:
# Standard Libraries
import os
import random
import warnings
import hashlib
import shutil

# Data Handling
import numpy as np
import pandas as pd

# Image Handling and Visualization
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
import cv2
from skimage.filters import gaussian
from sklearn.manifold import TSNE

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, array_to_img, load_img
from tensorflow.keras import models, layers
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import DenseNet121, InceptionV3
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D, Reshape


# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import hamming_loss, precision_recall_curve, brier_score_loss

# Miscellaneous
warnings.filterwarnings(action="ignore")
%matplotlib inline

In [8]:
# Define Colours
pal = ['#e3677d', '#bf4760', '#9c2544', '#7a002a', '#580011']

In [ ]:
# Base path of the dataset
train_data = '/kaggle/input/blood-cells/dataset2-master/dataset2-master/images/TRAIN'

# Blood cell classes
blood_cell_classes = sorted(os.listdir(train_data))

# Paths for the new split directories
split_train_dir = '/kaggle/working/split_train'
split_valid_dir = '/kaggle/working/split_valid'

In [ ]:
# Defining the function to split and move the blood cell images into their respective directories
def split_data(image_paths, labels, split_train_dir, split_valid_dir):
    if not os.path.exists(split_train_dir):
        os.makedirs(split_train_dir)
    if not os.path.exists(split_valid_dir):
        os.makedirs(split_valid_dir)

    # Creating class specific folders
    for blood_cell in blood_cell_classes:
        train_class_dir = os.path.join(split_train_dir, blood_cell)
        valid_class_dir = os.path.join(split_valid_dir, blood_cell)

        if not os.path.exists(train_class_dir):
            os.makedirs(train_class_dir)
        if not os.path.exists(valid_class_dir):
            os.makedirs(valid_class_dir)

    # Spliting dataset into training and validation
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        image_paths, labels, test_size=0.2, stratify=labels, random_state=42
    )

    # Moving training images to their respective class folders in split_train_dir
    for path, label in zip(train_paths, train_labels):
        shutil.copy(path, os.path.join(split_train_dir, label))

    # Moving validation images to their respective class folders in split_valid_dir
    for path, label in zip(val_paths, val_labels):
        shutil.copy(path, os.path.join(split_valid_dir, label))

In [ ]:
# Listing all images from the train directory
image_paths = []
labels = []

for label in blood_cell_classes:
    class_dir = os.path.join(train_data, label)
    for img in os.listdir(class_dir):
        image_paths.append(os.path.join(class_dir, img))
        labels.append(label)

In [ ]:
# Spliting and moving images into the respective class folders
split_data(image_paths, labels, split_train_dir, split_valid_dir)

In [ ]:
# Defining a function to display 3 blood cell images from each class
def display_images(folder_path, class_name, num_images=3):
    class_path = os.path.join(folder_path, class_name)
    images = os.listdir(class_path)[:num_images]
    plt.figure(figsize=(20, 10))
    for i, img_name in enumerate(images):

        # Get image path
        img_path = os.path.join(class_path, img_name)
        img = mpimg.imread(img_path)

        # Make sub plot
        plt.subplot(1, num_images, i+1)
        plt.imshow(img)

        # Add title
        plt.title(f'{class_name} {i+1}')
        plt.axis('off')
    plt.show()

In [ ]:
# Displaying 3 images from each blood class
for blood_cell in blood_cell_classes:
    print(f"Blood Cell : {blood_cell}")
    display_images(split_train_dir, blood_cell)

The image displays a grid of blood cells, categorized into four types: eosinophil, lymphocyte, monocyte, and neutrophil. There are three examples provided for each blood cell type. The eosinophils exhibit a bilobed nucleus and prominent red granules. Lymphocytes are characterized by a large, round nucleus and a small amount of cytoplasm. Monocytes, the largest of the white blood cells, possess a kidney-shaped nucleus. Neutrophils show a multi-lobed nucleus. The image serves as a visual guide for identifying different types of white blood cells based on their unique morphological features.

## Exploratory Data Analysis

In [ ]:
# Root path of directories
root_folder = '/kaggle/working/'

# Blood cell classes
blood_cell_classes = ['EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']

In [ ]:
# Path of the test blood cell images
base_folder = '/kaggle/input/blood-cells/dataset2-master/dataset2-master/images'
test_data = os.path.join(base_folder, 'TEST_SIMPLE')

In [ ]:
# Defining a function to count the number of images in each bloodclass
def count_images(folder_path):
    class_counts = {}
    for blood_cell in blood_cell_classes:
        class_path = os.path.join(folder_path, blood_cell)
        class_counts[blood_cell] = len(os.listdir(class_path))
    return class_counts

In [ ]:
# Checking data distribution
train_counts = count_images(split_train_dir)
test_counts = count_images(test_data)
valid_counts = count_images(split_valid_dir)

# Displaying the class distribution
print("Class distribution in train data:", train_counts)
print("Class distribution in valid data:", valid_counts)
print("Class distribution in test data:", test_counts)

The training set is well balanced with 1998 eosinophil, 1986 lymphocyte, 1982 monocyte and 1999 neutrophil samples. The validation set also maintains balance with 499 eosinophil, 497 lymphocyte, 496 monocyte and 500 neutrophil entries. However, the test set is highly imbalanced, containing only 13 eosinophil, 6 lymphocyte, 4 monocyte and 48 neutrophil records. This imbalance may lead to biased evaluation results, as performance on underrepresented classes could be underestimated compared to the more frequent neutrophil category.


In [ ]:
# Plotting pie charts to visuvalise the data distribution
train_counts = {'EOSINOPHIL': 1998, 'LYMPHOCYTE': 1986, 'MONOCYTE': 1982, 'NEUTROPHIL': 1999}
valid_counts = {'EOSINOPHIL': 499, 'LYMPHOCYTE': 497, 'MONOCYTE': 496, 'NEUTROPHIL': 500}

# Defining a function to plot pie chart
def plot_pie_chart(ax, data, title):
    labels = data.keys()
    sizes = data.values()
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=pal, wedgeprops=dict(edgecolor='black'), textprops={'fontsize': 13, 'color': 'white'})
    ax.set_title(title, fontsize=18)
    ax.axis('equal')

# Creating subplots
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plotting pie charts
plot_pie_chart(axs[0], train_counts, 'Class Distribution in Training Data')
plot_pie_chart(axs[1], valid_counts, 'Class Distribution in Validation Data')
plt.tight_layout()
plt.show()

The pie charts illustrate the distribution of four blood cell types in both the training and validation datasets. In the training data, Eosinophil and Neutrophil each constitute 25.1 percent of the total, while Lymphocyte and Monocyte each make up 24.9 percent. The validation data shows an identical distribution, with Eosinophil and Neutrophil at 25.1 percent, and Lymphocyte and Monocyte at 24.9 percent. This balanced class distribution across both datasets ensures an equitable representation of all four cell types for model training and evaluation.

In [ ]:
# Directories for the train, valid and test dataset
directories = {'Train': split_train_dir, 'Valid': split_valid_dir, 'Test': test_data}

# Defining a function to check image sizes
def check_image_sizes(folder_path):
    image_sizes = set()
    for blood_cell in blood_cell_classes:
        class_path = os.path.join(folder_path, blood_cell)
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            with Image.open(img_path) as img:
                # Add the image size  to the set
                image_sizes.add(img.size)

    return image_sizes

In [ ]:
# Checking size of images from train, valid and test datasets
for dataset, dir_path in directories.items():
    sizes = check_image_sizes(dir_path)
    if len(sizes) == 1:
        print(f"All images in {dataset} set have the same size: {sizes.pop()}")
    else:
        print(f"Images in {dataset} set have different sizes: {sizes}")

All images across the train, validation, and test sets are consistent in size, each being 320 by 240 pixels. This uniformity ensures compatibility during model training and evaluation without requiring additional resizing operations. Consistent dimensions also help maintain the aspect ratio and reduce preprocessing complexity. Having identical image sizes across datasets supports fair comparisons in performance evaluation and avoids distortions or scaling issues, thereby contributing to reliable learning and generalization of the model across different sets.


In [ ]:
# Number of images to sample
num_sampled_images = 100

# Defining a function to compute the color histograms
def compute_average_color_distribution(folder_path, num_samples):
    red_hist = np.zeros(256)
    green_hist = np.zeros(256)
    blue_hist = np.zeros(256)
    total_images = 0

    # Looping over each class folder
    for blood_cell in blood_cell_classes:
        class_path = os.path.join(folder_path, blood_cell)
        images = os.listdir(class_path)

        # Randomly selecting images to process
        sampled_images = random.sample(images, min(num_samples, len(images)))

        # Processing each sampled image
        for img_name in sampled_images:
            img_path = os.path.join(class_path, img_name)
            with Image.open(img_path) as img:
                img_array = np.array(img)

                # Flattening RGB channels and computing histograms
                red_hist += np.histogram(img_array[:, :, 0], bins=256, range=(0, 256))[0]
                green_hist += np.histogram(img_array[:, :, 1], bins=256, range=(0, 256))[0]
                blue_hist += np.histogram(img_array[:, :, 2], bins=256, range=(0, 256))[0]

                total_images += 1

    # Averaging histograms by dividing by the total number of images processed
    red_hist /= total_images
    green_hist /= total_images
    blue_hist /= total_images

    return red_hist, green_hist, blue_hist

In [ ]:
# Plotting averaged color distribution
def plot_average_color_distribution(red_hist, green_hist, blue_hist):
    plt.figure(figsize=(20, 5))

    # Make 3 subplots

    plt.subplot(1, 3, 1)
    plt.plot(red_hist, color='red')

    # Add title
    plt.title('Average Red Channel Distribution')

    plt.subplot(1, 3, 2)
    plt.plot(green_hist, color='green')
    # Add title
    plt.title('Average Green Channel Distribution')

    plt.subplot(1, 3, 3)
    plt.plot(blue_hist, color='blue')
    # Add title
    plt.title('Average Blue Channel Distribution')

    # Show plot
    plt.tight_layout()
    plt.show()

In [ ]:
# Computing average color distribution for the Train data
red_hist, green_hist, blue_hist = compute_average_color_distribution(split_train_dir, num_sampled_images)

# Plotting average color distribution
plot_average_color_distribution(red_hist, green_hist, blue_hist)

The histograms display the average distribution of Red, Green, and Blue color channels. The red channel shows a prominent peak near the lower values, reaching a count over 8000, and another smaller peak around 200. The green channel has its highest peak also near the lower values, approaching 8000, with two additional peaks around 175 and 225. The blue channel exhibits a large peak near zero, exceeding a count of 8000, and a subsequent peak around 225. These distributions highlight the varying color intensity levels in the image dataset.

In [ ]:
base_folder = '/kaggle/input/blood-cells/dataset2-master/dataset2-master/images'
train_folder = os.path.join(base_folder, "TRAIN")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()  # flatten to 1D for easy indexing

for i, cls in enumerate(blood_cell_classes):
    cls_path = os.path.join(train_folder, cls)
    random_img = random.choice(os.listdir(cls_path))
    img_path = os.path.join(cls_path, random_img)


    img = cv2.imread(img_path)

    # Convert image to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Make histogram plot
    axes[i].hist(gray.ravel(), bins=50, color=pal[i % len(pal)], alpha=0.8)

    # Add title and labels
    axes[i].set_title(f"{cls} (Grayscale)", fontsize=18)
    axes[i].set_xlabel("Pixel Intensity", fontsize=16)
    axes[i].set_ylabel("Frequency", fontsize=16)
    axes[i].tick_params(axis='both', labelsize=12)

# Add main title
plt.suptitle("Grayscale Intensity Distributions (TRAIN Set)", fontsize=20, y=1.02)

# Show plot
plt.tight_layout()
plt.show()

The image displays grayscale intensity distributions for four types of blood cells in the training set. Eosinophil and Lymphocyte show a large peak at low pixel intensity and another broad peak centered around a value of 200. Monocyte exhibits a similar distribution with a large peak at low intensity and another peak around 200, though with higher frequencies. Neutrophil also has a very high frequency peak near zero and a broader distribution of higher intensity values, with a major peak around 200. Overall, all four cell types share a common pattern of high frequencies at both low and high pixel intensities.

In [ ]:
# Convert image to dataframe
r, g, b = cv2.split(img)
df = pd.DataFrame({"Red": r.flatten(), "Green": g.flatten(), "Blue": b.flatten()})

# Plot heatmap
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap( df.corr(),annot=True,annot_kws={"size": 13},cmap="Reds",cbar=True,ax=ax, linewidths=1, linecolor="black")

# Add title
ax.set_title("RGB Channel Correlation", fontsize=18)

# Show plot
plt.show()

The heat map illustrates the correlation between the Red, Green, and Blue color channels. The diagonal elements, representing the correlation of each channel with itself, are all 1. The Red and Green channels show a strong positive correlation of 0.92. The Red and Blue channels have a high correlation of 0.95. The strongest correlation is observed between the Green and Blue channels, with a value of 0.98. These high correlation values suggest that the color information across the different channels is highly interdependent in the image dataset.

In [ ]:
# Brightness distribution
brightness = []

for cls in blood_cell_classes:
    cls_path = os.path.join(train_folder, cls)
    for img_name in os.listdir(cls_path)[:300]:  # Sample images
        img = cv2.imread(os.path.join(cls_path, img_name))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness.append((cls, gray.mean()))

df_brightness = pd.DataFrame(brightness, columns=["Class", "Brightness"])

plt.figure(figsize=(8,6))
sns.boxplot(x="Class", y="Brightness", data=df_brightness, palette=pal[:4])

# Add title and labels
plt.title("Brightness Distribution Across Classes", fontsize=18)
plt.xlabel("Cell Type", fontsize=16)
plt.ylabel("Mean Brightness", fontsize=16)

# Add ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Show plot
plt.show()

The box plot displays the brightness distribution for four blood cell types. Eosinophil has a median brightness of approximately 165, with a range from around 145 to 188. Lymphocyte shows a higher median brightness of about 169, with a range from 145 to 192. Monocyte has a median of approximately 166, spanning from 143 to 190. Neutrophil has the highest median brightness around 168, with a range from 145 to 195. All four cell types have a similar brightness range, but Lymphocyte and Neutrophil appear slightly brighter on average.

In [ ]:
# Edge Detection
fig, axes = plt.subplots(1, 4, figsize=(16,4))

for i, cls in enumerate(blood_cell_classes):
    cls_path = os.path.join(train_folder, cls)
    img_name = random.choice(os.listdir(cls_path))

    # Convert image to grayscale
    img = cv2.imread(os.path.join(cls_path, img_name), cv2.IMREAD_GRAYSCALE)

    # Apply Canny Edge
    edges = cv2.Canny(img, 100, 200)

    axes[i].imshow(edges, cmap="gray")
    axes[i].set_title(f"{cls} (Edges)", fontsize=15)
    axes[i].axis("off")


# Add title
plt.suptitle("Edge Detection in Each Class Image", fontsize=20, y=1.05)

# Show plot
plt.tight_layout()
plt.show()

The image displays the results of edge detection for four blood cell types. Eosinophil and Monocyte show distinct inner circular shapes. Lymphocyte and Neutrophil exhibit more complex outer contours and inner edge patterns. These images highlight how edge detection captures the structural differences of each cell type.

In [ ]:
# Prepare features
features = []
labels = []

for cls in blood_cell_classes:
    cls_path = os.path.join(train_folder, cls)
    for img_name in os.listdir(cls_path)[:200]:  # Sample 200 per class
        img = cv2.imread(os.path.join(cls_path, img_name))
        img = cv2.resize(img, (64,64))

        # Use color histogram as features
        hist = cv2.calcHist([img], [0,1,2], None, [8,8,8], [0,256,0,256,0,256])
        hist = cv2.normalize(hist, hist).flatten()

        features.append(hist)
        labels.append(cls)

features = np.array(features)

# Run  t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_features = tsne.fit_transform(features)

# Make plot
plt.figure(figsize=(8,6))
for i, cls in enumerate(blood_cell_classes):
    idx = np.array(labels) == cls
    plt.scatter(tsne_features[idx,0], tsne_features[idx,1],
                color=pal[i], alpha=0.6, label=cls, edgecolor="black", s=50)

# Add title and labels
plt.title("t-SNE Visualisation of Image Features", fontsize=18)
plt.xlabel("t-SNE Dim 1", fontsize=16)
plt.ylabel("t-SNE Dim 2", fontsize=16)

# Add legend
plt.legend(fontsize=12)

# Show plot
plt.show()

The t-SNE plot visualises the image features of four blood cell types in a two-dimensional space. The plot reveals significant overlap between the clusters, indicating that the features are not easily separable. Eosinophil, Lymphocyte, Monocyte, and Neutrophil points are mixed together across the entire plot, without forming distinct, isolated groups. This suggests that the image features of these cell types are quite similar, which could pose a challenge for a classification model relying on these features alone.

In [ ]:
# Defining paths for train and valid data
paths = {
    'TRAIN': '/kaggle/input/blood-cells/dataset2-master/dataset2-master/images/TRAIN',
    'VALID': '/kaggle/input/blood-cells/dataset2-master/dataset2-master/images/TEST'
}


In [ ]:
# Function which generate hash for an blood cell image
def generate_image_hash(img_path):
    with Image.open(img_path) as img:
        img = img.convert('RGB')
        hash_value = hashlib.md5(img.tobytes()).hexdigest()
    return hash_value

In [ ]:
# Function which finds duplicate blood cell images
def find_duplicates(folder_path):
    hash_dict = {}
    duplicates = []

    # Looping over each class folder
    for blood_cell in blood_cell_classes:
        class_path = os.path.join(folder_path, blood_cell)
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            img_hash = generate_image_hash(img_path)

            if img_hash in hash_dict:
                duplicates.append((img_path, hash_dict[img_hash]))
            else:
                hash_dict[img_hash] = img_path

    return duplicates

In [ ]:
# Finding duplicates in train and valid datasets
for dataset, path in paths.items():
    print(f"\nChecking for duplicates in {dataset} dataset...")
    duplicates = find_duplicates(path)

    if duplicates:
        print(f"Found {len(duplicates)} duplicate(s) in {dataset} dataset.")
        # Optional: Print duplicate pairs
        for dup in duplicates:
            print(f"Duplicate pair: {dup[0]} and {dup[1]}")
    else:
        print(f"No duplicates found in {dataset} dataset.")

The duplicate check confirms that both the train and validation datasets contain no repeated images. This indicates a clean dataset with unique samples, which helps ensure unbiased learning during model training and reliable performance during validation. The absence of duplicates also reduces the risk of data leakage, where repeated instances could artificially inflate performance metrics. Overall, the datasets appear well-prepared and trustworthy for building a robust classification model.


## Data Preprocessing

In [ ]:
# Define a function to display one sample image from each class
def show_samples_from_each_class(directory, num_classes):
    # Get sorted list of class names (folder names)
    blood_cell_classes = sorted(os.listdir(directory))

    # Create subplots  for displaying images
    fig, axes = plt.subplots(1, num_classes, figsize=(15, 5))

    for i, blood_cell in enumerate(blood_cell_classes):
        # Build the full path to the current class folder
        class_folder = os.path.join(directory, blood_cell)

        # Get list of image files in the current class folder
        image_files = sorted(os.listdir(class_folder))

        if image_files:
            # Use the first image file found in the folder
            image_file = image_files[0]
            image_path = os.path.join(class_folder, image_file)

            # Read and convert the image from BGR to RGB
            image = cv2.imread(image_path)
            if image is not None:
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                axes[i].imshow(image_rgb)
                # Add title
                axes[i].set_title(blood_cell, fontsize=18)
            else:
                print(f"Warning: Image {image_path} could not be read.")
        else:
            print(f"Warning: No images found in {class_folder}.")

        # Hide axis ticks for a cleaner look
        axes[i].axis('off')
    plt.tight_layout()
    # Show plot
    plt.show()


In [ ]:
# Calling a defined function to display a blood cell images
num_classes = len(os.listdir(split_train_dir))
show_samples_from_each_class(split_train_dir, num_classes)

### HSV Masking and Morphological Operations

HSV masking isolates specific colour ranges by converting images into hue, saturation, and value space for better segmentation. Morphological operations refine these masks by removing noise and enhancing object boundaries. They are used together to improve accuracy in detecting and highlighting relevant regions within biomedical images.


In [ ]:
# Defining a function for applying HSV Masking
def apply_hsv_mask(image):
    # Converting image to HSV color space
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    lower_purple = np.array([110, 50, 50])
    upper_purple = np.array([170, 255, 255])

    # Creating mask for purple color
    mask = cv2.inRange(hsv_image, lower_purple, upper_purple)
    return mask

In [ ]:
def preprocess_step_1(source_dir, target_dir):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    for blood_cell in blood_cell_classes:
        class_folder = os.path.join(source_dir, blood_cell)
        target_class_folder = os.path.join(target_dir, blood_cell)

        if not os.path.exists(target_class_folder):
            os.makedirs(target_class_folder)

        image_files = sorted(os.listdir(class_folder))

        for image_file in image_files:
            image_path = os.path.join(class_folder, image_file)
            image = cv2.imread(image_path)

            if image is not None:
                # Applying HSV Masking
                mask = apply_hsv_mask(image)

                # Converting refined mask to grayscale
                mask_gray = cv2.cvtColor(cv2.bitwise_and(image, image, mask=mask), cv2.COLOR_BGR2GRAY)

                # Saving preprocessed image
                target_image_path = os.path.join(target_class_folder, image_file)
                cv2.imwrite(target_image_path, mask_gray)
            else:
                print(f"Warning: Image {image_path} could not be read.")

    print(f'Preprocessed images (Step 1) saved to {target_dir}')

In [ ]:
def display_samples_from_each_class(directory, num_classes):
    fig, axes = plt.subplots(1, num_classes, figsize=(15, 5))

    for i, blood_cell in enumerate(blood_cell_classes):
        class_folder = os.path.join(directory, blood_cell)
        image_files = sorted(os.listdir(class_folder))

        if image_files:

            # Take the first image in each class folder
            image_file = image_files[0]
            image_path = os.path.join(class_folder, image_file)
            image = cv2.imread(image_path)

            # Convert to RGB for display
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            axes[i].imshow(image_rgb)

            # Add title
            axes[i].set_title(blood_cell, fontsize=18)
            axes[i].axis('off')

    # Show plot
    plt.tight_layout()
    plt.show()

In [ ]:
# Displaying training images after applying HSV masking technique
preprocessed_train_step1 = '/kaggle/working/train_step1'
preprocess_step_1(split_train_dir, preprocessed_train_step1)
display_samples_from_each_class(preprocessed_train_step1, num_classes)

In [ ]:
# Displaying validation images after applying HSV masking technique
preprocessed_valid_step1 = '/kaggle/working/valid_step1'
preprocess_step_1(split_valid_dir, preprocessed_valid_step1)
display_samples_from_each_class(preprocessed_valid_step1, num_classes)

In [ ]:
# Displaying testing images after applying HSV masking technique
preprocessed_test_step1 = '/kaggle/working/test_step1'
preprocess_step_1(test_data, preprocessed_test_step1)
display_samples_from_each_class(preprocessed_test_step1, num_classes)

### Contrast Enhancement

Contrast enhancement is a technique used to improve the visibility of features in an image by adjusting intensity levels. It highlights subtle details that may otherwise be difficult to distinguish. In biomedical imaging, it is applied to ensure clearer differentiation of cell structures for accurate analysis.


In [ ]:
# Defining a function to apply Contrast Enhancement technique
def increase_contrast(image):
    # Converting image to LAB color space
    lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

    # Splitting LAB image into channels
    l_channel, a_channel, b_channel = cv2.split(lab_image)

    # Applying CLAHE to L-channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_l_channel = clahe.apply(l_channel)

    # Merging enhanced L-channel back with a and b channels
    enhanced_lab_image = cv2.merge((enhanced_l_channel, a_channel, b_channel))

    # Converting LAB image back to BGR color space
    contrast_enhanced_image = cv2.cvtColor(enhanced_lab_image, cv2.COLOR_LAB2BGR)
    return contrast_enhanced_image

In [ ]:
def preprocess_step_2_contrast(source_dir, target_dir):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    for blood_cell in blood_cell_classes:
        class_folder = os.path.join(source_dir, blood_cell)
        target_class_folder = os.path.join(target_dir, blood_cell)

        if not os.path.exists(target_class_folder):
            os.makedirs(target_class_folder)

        # Sort image files
        image_files = sorted(os.listdir(class_folder))

        for image_file in image_files:
            image_path = os.path.join(class_folder, image_file)
            image = cv2.imread(image_path)

            if image is not None:
                # Applying Contrast Enhancement
                enhanced_image = increase_contrast(image)

                # Savinge contrast enhanced image
                target_image_path = os.path.join(target_class_folder, image_file)
                cv2.imwrite(target_image_path, enhanced_image)
            else:
                print(f"Warning: Image {image_path} could not be read.")

    print(f'Preprocessed images (Step 2 - Contrast) saved to {target_dir}')

In [ ]:
# Displaying training images after applying Contrast Enhancement technique
preprocessed_train_step2_contrast = '/kaggle/working/train_step2_contrast'
preprocess_step_2_contrast(preprocessed_train_step1, preprocessed_train_step2_contrast)
display_samples_from_each_class(preprocessed_train_step2_contrast, num_classes)

In [ ]:
# Displaying validation images after applying Contrast Enhancement technique
preprocessed_valid_step2_contrast = '/kaggle/working/valid_step2_contrast'
preprocess_step_2_contrast(preprocessed_valid_step1, preprocessed_valid_step2_contrast)
display_samples_from_each_class(preprocessed_valid_step2_contrast, num_classes)

In [ ]:
# Displaying testing images after applying Contrast Enhancement technique
preprocessed_test_step2_contrast = '/kaggle/working/test_step2_contrast'
preprocess_step_2_contrast(preprocessed_test_step1, preprocessed_test_step2_contrast)
display_samples_from_each_class(preprocessed_test_step2_contrast, num_classes)

### Sharpening

Sharpening is an image processing technique that enhances edges and fine details by increasing local contrast. It makes boundaries between different regions more distinct, improving clarity. In biomedical imaging, sharpening helps highlight important cellular structures, aiding precise visual analysis and classification.

In [ ]:
# Defining a function for applying Sharpening technique on images
def sharpen_image(image):
    # Creating sharpening kernel
    kernel = np.array([[0, -1, 0],
                       [-1, 5,-1],
                       [0, -1, 0]])

    # Applying kernel to the input image
    sharpened_image = cv2.filter2D(image, -1, kernel)

    return sharpened_image

In [ ]:
def preprocess_step_3_sharpen(source_dir, target_dir):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    for blood_cell in blood_cell_classes:
        class_folder = os.path.join(source_dir, blood_cell)
        target_class_folder = os.path.join(target_dir, blood_cell)

        if not os.path.exists(target_class_folder):
            os.makedirs(target_class_folder)

        # Sort image files
        image_files = sorted(os.listdir(class_folder))

        for image_file in image_files:
            image_path = os.path.join(class_folder, image_file)
            image = cv2.imread(image_path)

            if image is not None:
                # Apply sharpening
                sharpened_image = sharpen_image(image)

                # Save the sharpened image
                target_image_path = os.path.join(target_class_folder, image_file)
                cv2.imwrite(target_image_path, sharpened_image)
            else:
                print(f"Warning: Image {image_path} could not be read.")

    print(f'Preprocessed images (Step 3 - Sharpened) saved to {target_dir}')

In [ ]:
# Displaying training images after applying Sharpening technique
preprocessed_train_step3_sharpen = '/kaggle/working/train_step3_sharpen'
preprocess_step_3_sharpen(preprocessed_train_step2_contrast, preprocessed_train_step3_sharpen)
display_samples_from_each_class(preprocessed_train_step3_sharpen, num_classes)

In [ ]:
# Displaying validation images after applying Sharpening technique
preprocessed_valid_step3_sharpen = '/kaggle/working/valid_step3_sharpen'
preprocess_step_3_sharpen(preprocessed_valid_step2_contrast, preprocessed_valid_step3_sharpen)
display_samples_from_each_class(preprocessed_valid_step3_sharpen, num_classes)

In [ ]:
# Displaying testing images after applying Sharpening technique
preprocessed_test_step3_sharpen = '/kaggle/working/test_step3_sharpen'
preprocess_step_3_sharpen(preprocessed_test_step2_contrast, preprocessed_test_step3_sharpen)
display_samples_from_each_class(preprocessed_test_step3_sharpen, num_classes)

## Data Augmentation

Data augmentation is the process of artificially increasing dataset diversity by applying transformations such as rotation, flipping, scaling, or colour adjustments. It helps prevent overfitting by exposing the model to varied representations of the same data. In biomedical imaging, augmentation ensures robustness and improves generalisation when training with limited samples.


In [ ]:
# Paths of the preprocessed training and validation blood cell images
train_data_prep = '/kaggle/working/train_step3_sharpen'
valid_data_prep = '/kaggle/working/valid_step3_sharpen'

In [ ]:
# Creating ImageDataGenerators with Augmentation for Training images
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='nearest'
)

# No augmentations for validation images
valid_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
# Train Generator with Augmentation
train_generator = train_datagen.flow_from_directory(
    train_data_prep,
    target_size=(224, 224),
    color_mode='rgb',
    batch_size=16,
    class_mode='categorical',
    shuffle=True
)

# Validation Generator without augmentation
valid_generator = valid_datagen.flow_from_directory(
    valid_data_prep,
    target_size=(224, 224),
    color_mode='rgb',
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# Preparing test dataset directory
test_data_dir = '/kaggle/working/test_step3_sharpen'

# Creating ImageDataGenerator for the test dataset
test_datagen = ImageDataGenerator(rescale=1./255)

# Test Generator without augmentation
test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(224, 224),
    color_mode='rgb',
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

## Model Building

In [ ]:
def plot_learning_curves(history, suptitle="Learning Curves"):
    """
    Plots training and validation accuracy and loss from model training history.

    Parameters:
    history : History object returned by model.fit()
    suptitle : str, main title for the plot
    """
    # Prepare data
    epochs = history.epoch
    train_accuracy = history.history['accuracy']
    val_accuracy = history.history['val_accuracy']
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

    # Plot accuracy
    ax1.plot(epochs, train_accuracy, label='Training Accuracy', color='#e3677d', marker='o')
    ax1.plot(epochs, val_accuracy, label='Validation Accuracy', color='#580011', marker='o')

    # Add title and labels
    ax1.set_title('Training and Validation Accuracy', fontsize=18)
    ax1.set_xlabel('Epochs', fontsize=16)
    ax1.set_ylabel('Accuracy', fontsize=16)

    # Add legend
    ax1.legend()

    # Plot loss
    ax2.plot(epochs, train_loss, label='Training Loss', color='#e3677d', marker='o')
    ax2.plot(epochs, val_loss, label='Validation Loss', color='#580011', marker='o')

    # Add title and labels
    ax2.set_title('Training and Validation Loss', fontsize=18)
    ax2.set_xlabel('Epochs', fontsize=16)
    ax2.set_ylabel('Loss', fontsize=16)

    # Add legend
    ax2.legend()

    # Add main suptitle
    plt.suptitle(suptitle, fontsize=20, y=1.05)

    # Show plot
    plt.tight_layout()
    plt.show()

### BloodCnn Model

A Convolutional Neural Network, or CNN, is a deep learning model designed to automatically extract spatial features from images. It uses layers like convolution, pooling, and fully connected layers to capture patterns such as edges, textures, and shapes. In biomedical imaging, CNNs are effective for classifying cells or detecting abnormalities due to their ability to learn hierarchical visual features without manual feature engineering.


In [ ]:
# Defining CNN Model for the blood cell classification task
def create_cnn_model():
    cnn_model = models.Sequential([
        # Initial depthwise separable convolution layer with large kernel and stride
        layers.SeparableConv2D(128, (8, 8), strides=(3, 3), activation='relu', input_shape=(224, 224, 3)),
        layers.BatchNormalization(),

        # Additional separable convolution with smaller kernel, preserving dimensions
        layers.SeparableConv2D(128, (5, 5), strides=(1, 1), activation='relu', padding="same"),
        layers.BatchNormalization(),

        # Increase feature depth and keep spatial resolution
        layers.SeparableConv2D(256, (3, 3), strides=(1, 1), activation='relu', padding="same"),
        layers.BatchNormalization(),

        # Pointwise (1x1) convolution to further process features
        layers.SeparableConv2D(256, (1, 1), strides=(1, 1), activation='relu', padding="same"),
        layers.BatchNormalization(),

        # Downsample feature maps to reduce spatial size
        layers.MaxPool2D(pool_size=(2, 2)),

        # Further convolutional processing with more filters
        layers.SeparableConv2D(512, (3, 3), activation='relu', padding="same"),
        layers.BatchNormalization(),

        # Additional max pooling for further downsampling
        layers.MaxPool2D(pool_size=(2, 2)),

        # Deep convolutional layer for capturing complex patterns
        layers.SeparableConv2D(512, (3, 3), activation='relu', padding="same"),
        layers.BatchNormalization(),

        # Global average pooling to reduce each feature map to a single value
        layers.GlobalAveragePooling2D(),

        # Fully connected layer with L2 regularization and dropout for regularization
        layers.Dense(1024, activation='relu', kernel_regularizer=l2(0.001)),
        layers.Dropout(0.5),

        # Second dense layer for deeper representation learning
        layers.Dense(1024, activation='relu', kernel_regularizer=l2(0.001)),
        layers.Dropout(0.5),

        # Output layer for 4-class classification with softmax activation
        layers.Dense(4, activation='softmax')
    ])

    return cnn_model

In [ ]:
# CNN Model Training and Evaluation
blood_cnn_model = create_cnn_model()
blood_cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# fitting a CNN model
history_blood_cnn = blood_cnn_model.fit(train_generator, epochs=10, validation_data=valid_generator)

In [ ]:
# Get true labels and predicted labels for cnn model
validation_steps = valid_generator.samples // valid_generator.batch_size

# List to store true and predicted labels
true_labels = []
predicted_labels = []

# Iterate over the validation generator
for i in range(validation_steps):
    x_batch, y_batch = valid_generator[i]
    true_labels.extend(np.argmax(y_batch, axis=1))

    # Predict Labels using the Model
    predictions = blood_cnn_model.predict(x_batch, verbose=0)
    predicted_labels.extend(np.argmax(predictions, axis=1))

#  Convert to NumPy Arrays
cnn_true_labels = np.array(true_labels)
cnn_predicted_labels = np.array(predicted_labels)

blood_cnn_pred_probs = blood_cnn_model.predict(valid_generator, steps=validation_steps, verbose=0)
blood_cnn_y_true_onehot = np.zeros_like(blood_cnn_pred_probs)
blood_cnn_y_true_onehot[np.arange(len(cnn_true_labels)), cnn_true_labels] = 1

In [ ]:
# Compute Metrics
blood_cnn_accuracy = accuracy_score(cnn_true_labels, cnn_predicted_labels)
blood_cnn_precision = precision_score(cnn_true_labels, cnn_predicted_labels, average='weighted')
blood_cnn_recall = recall_score(cnn_true_labels, cnn_predicted_labels, average='weighted')
blood_cnn_f1 = f1_score(cnn_true_labels, cnn_predicted_labels, average='weighted')
blood_cnn_hamming = hamming_loss(cnn_true_labels, cnn_predicted_labels)
blood_cnn_brier_score = np.mean(np.sum((blood_cnn_pred_probs - blood_cnn_y_true_onehot) ** 2, axis=1))

In [ ]:
print("BloodCnn Model Metrics Evaluation Results")
print("-" * 30)
print(f"Accuracy: {blood_cnn_accuracy:.4f}")
print(f"Precision: {blood_cnn_precision:.4f}")
print(f"Recall: {blood_cnn_recall:.4f}")
print(f"F1-score: {blood_cnn_f1:.4f}")
print(f"Hamming Loss: {blood_cnn_hamming:.4f}")
print(f"Brier Score: {blood_cnn_brier_score:.4f}")
print("-" * 30)

The BloodCNN model demonstrates strong performance with an accuracy of 0.9032, indicating that over 90 percent of predictions are correct. Precision is 0.9201, showing high correctness among predicted positive cases, while recall of 0.9032 reflects effective detection of actual positives. The F1-score of 0.9012 balances precision and recall. Low Hamming Loss of 0.0968 and a Brier Score of 0.1448 indicate minimal classification errors and good probability calibration, making the model reliable for blood cell classification tasks.


In [ ]:
# Classification Report
report = classification_report(cnn_true_labels, cnn_predicted_labels, target_names=valid_generator.class_indices.keys())
print(report)

In [ ]:

# Generate predictions and true labels if not already done
blood_cnn_predictions = blood_cnn_model.predict(valid_generator, verbose=1)
blood_cnn_predicted_labels = np.argmax(blood_cnn_predictions, axis=1)
blood_cnn_true_labels = valid_generator.classes

# Calculate the confusion matrix
blood_cnn_conf_matrix = confusion_matrix(blood_cnn_true_labels, blood_cnn_predicted_labels)

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(blood_cnn_conf_matrix,annot=True, fmt='d',cmap="Reds", cbar=True, ax=ax, linewidths=1, linecolor="black",  annot_kws={"size": 13}, xticklabels=blood_cell_classes, yticklabels=blood_cell_classes )

# Add title and labels
ax.set_title('Confusion Matrix for BloodCnn Model', fontsize=18)
ax.set_xlabel('Predicted Label', fontsize=16)
ax.set_ylabel('True Label', fontsize=16)

# Show plot
plt.show()

The confusion matrix for the BloodCnn model shows its classification performance. The model correctly identified 345 Eosinophils, 496 Lymphocytes, 465 Monocytes, and 494 Neutrophils. However, it misclassified 146 Eosinophils as Neutrophils and 6 as Lymphocytes, and 2 as Monocytes. There were also 30 Monocytes wrongly predicted as Lymphocytes. These results highlight a specific challenge in differentiating Eosinophils from Neutrophils, while the model performed well in classifying Lymphocytes and Monocytes. The high numbers of correctly predicted labels indicate strong overall performance.

In [ ]:
plt.figure(figsize=(8, 6))

# Precision-Recall Curves for CNN model
for i in range(len(blood_cell_classes)):
    y_true_cnn = (blood_cnn_true_labels == i).astype(int)
    y_pred_cnn = blood_cnn_predictions[:, i]
    precision_cnn, recall_cnn, _ = precision_recall_curve(y_true_cnn, y_pred_cnn)

    # Plot with color
    plt.plot(recall_cnn, precision_cnn, label=f'Class {blood_cell_classes[i]} CNN PR Curve',
             color=pal[i])

# Add title and labels
plt.title('Precision-Recall Curves for BloodCNN - All Classes', fontsize=18)
plt.xlabel('Recall', fontsize=16)
plt.ylabel('Precision', fontsize=16)

# Add legend
plt.legend(loc='best')

# Show plot
plt.show()

The precision-recall curves show the model's performance for each class. The Lymphocyte and Monocyte classes have consistently high precision above 0.9. The Neutrophil class also shows high precision, dropping below 0.9 only at high recall. The Eosinophil curve has the lowest precision overall, falling sharply after a recall of about 0.8.

In [ ]:
plot_learning_curves(history_blood_cnn, suptitle="BloodCnn Training Performance on Blood Cell Dataset")

The plot shows the BloodCnn model's training and validation performance. The training accuracy rises steadily, reaching over 0.85 by epoch 5 and peaking at 0.9. Validation accuracy fluctuates, with peaks around 0.86 at epoch 3 and 0.89 at epoch 5. Training loss decreases from 1.75 to 0.45. Validation loss also decreases, though with more fluctuations, reaching a low of 0.4 before a slight increase. This suggests the model is generally learning well, but the fluctuations in validation performance indicate potential overfitting or instability during training.

### BloodDenseNet121

Fine-tuning DenseNet121 involves taking a pre-trained DenseNet121 model and adjusting its weights on a new, specific dataset while retaining learned features from large-scale data. This approach leverages existing knowledge to improve performance on tasks with limited data. It reduces training time and computational cost compared to training from scratch. Fine-tuning helps the model adapt to domain-specific patterns, such as cell morphology in biomedical images. Overall, it enhances accuracy and generalisation for specialised classification tasks.


In [ ]:
def create_finetuned_densenet(input_shape=(224, 224, 3), num_classes=4, unfreeze_layers=50):
    # Load DenseNet121 base model with ImageNet weights
    base_model = DenseNet121(include_top=False, weights='imagenet', input_shape=input_shape)

    # Freeze all layers initially
    base_model.trainable = False

    # Global average pooling + dense layers
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(1024, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(num_classes, activation='softmax')(x)

    # Create the model
    model = models.Model(inputs=base_model.input, outputs=output)

    # Unfreeze the top `unfreeze_layers` layers of DenseNet121 for fine-tuning
    for layer in base_model.layers[-unfreeze_layers:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

    return model

In [ ]:
# DenseNet Model Training and Evaluation
blood_densenet_model = create_finetuned_densenet(unfreeze_layers=50)
blood_densenet_model.compile(optimizer=Adam(learning_rate=1e-5),loss='categorical_crossentropy',metrics=['accuracy'] )

In [ ]:
# Train the model
history_densenet = blood_densenet_model.fit(train_generator, epochs=10,validation_data=valid_generator)

In [ ]:
# List to store true and predicted labels
true_labels = []
predicted_labels = []

# Iterate over the validation generator
for i in range(validation_steps):
    x_batch, y_batch = valid_generator[i]
    true_labels.extend(np.argmax(y_batch, axis=1))

    # Predict Labels using the Model
    predictions = blood_densenet_model.predict(x_batch, verbose=0)
    predicted_labels.extend(np.argmax(predictions, axis=1))

#  Convert to NumPy Arrays
densenet_true_labels = np.array(true_labels)
densenet_predicted_labels = np.array(predicted_labels)

blood_densenet_pred_probs = blood_densenet_model.predict(valid_generator, steps=validation_steps, verbose=0)
blood_densenet_y_true_onehot = np.zeros_like(blood_densenet_pred_probs)
blood_densenet_y_true_onehot[np.arange(len(densenet_true_labels)), densenet_true_labels] = 1

In [ ]:
# Compute Metrics
blood_densenet_accuracy = accuracy_score(densenet_true_labels, densenet_predicted_labels)
blood_densenet_precision = precision_score(densenet_true_labels, densenet_predicted_labels, average='weighted')
blood_densenet_recall = recall_score(densenet_true_labels, densenet_predicted_labels, average='weighted')
blood_densenet_f1 = f1_score(densenet_true_labels, densenet_predicted_labels, average='weighted')
blood_densenet_hamming = hamming_loss(densenet_true_labels, densenet_predicted_labels)
blood_densenet_brier_score = np.mean(np.sum((blood_densenet_pred_probs - blood_densenet_y_true_onehot) ** 2, axis=1))

In [ ]:
print("BloodDenseNet121 Model Metrics Evaluation Results")
print("-" * 30)
print(f"Accuracy: {blood_densenet_accuracy:.4f}")
print(f"Precision: {blood_densenet_precision:.4f}")
print(f"Recall: {blood_densenet_recall:.4f}")
print(f"F1-score: {blood_densenet_f1:.4f}")
print(f"Hamming Loss: {blood_densenet_hamming:.4f}")
print(f"Brier Score: {blood_densenet_brier_score:.4f}")
print("-" * 30)

The BloodDenseNet121 model achieves an accuracy of 0.8412, indicating that about 84 percent of predictions are correct. Precision is 0.8438, showing reliable positive predictions, while recall of 0.8412 reflects effective identification of actual positives. The F1-score of 0.8363 balances precision and recall, indicating consistent performance. Higher Hamming Loss of 0.1588 and Brier Score of 0.2256 compared to other models suggest slightly more classification errors and less precise probability estimates. Overall, the model performs well but is less accurate than BloodCNN for blood cell classification.


In [ ]:
# Classification Report
report = classification_report(densenet_true_labels, densenet_predicted_labels, target_names=valid_generator.class_indices.keys())
print(report)

In [ ]:

# Generate predictions and true labels if not already done
blood_densenet_predictions = blood_densenet_model.predict(valid_generator, verbose=1)
blood_densenet_predicted_labels = np.argmax(blood_densenet_predictions, axis=1)
blood_densenet_true_labels = valid_generator.classes

# Calculate the confusion matrix
blood_densenet_conf_matrix = confusion_matrix(blood_densenet_true_labels, blood_densenet_predicted_labels)

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(blood_densenet_conf_matrix,annot=True, fmt='d',cmap="Reds", cbar=True, ax=ax, linewidths=1, linecolor="black",  annot_kws={"size": 13}, xticklabels=blood_cell_classes, yticklabels=blood_cell_classes )

# Add title and labels
ax.set_title('Confusion Matrix for BloodDenseNet121 Model', fontsize=18)
ax.set_xlabel('Predicted Label', fontsize=16)
ax.set_ylabel('True Label', fontsize=16)

# Show plot
plt.show()

The confusion matrix for the BloodDenseNet121 model shows its performance on four blood cell classes. The model correctly identified 368 Eosinophils, 496 Lymphocytes, 472 Monocytes, and 340 Neutrophils. However, it showed significant misclassification. Notably, 103 Neutrophils were misclassified as Eosinophils. Additionally, 58 Eosinophils were incorrectly predicted as Lymphocytes, and 42 were classified as Neutrophils. Similarly, 33 Neutrophils were wrongly classified as Lymphocytes, and 24 as Monocytes. This indicates that while the model performs well on Lymphocytes and Monocytes, it struggles with differentiating Eosinophils and Neutrophils.

In [ ]:
plt.figure(figsize=(8, 6))

# Precision-Recall Curves for Densenet model
for i in range(len(blood_cell_classes)):
    y_true_densenet = (blood_densenet_true_labels == i).astype(int)
    y_pred_densenet = blood_densenet_predictions[:, i]
    precision_densenet, recall_densenet, _ = precision_recall_curve(y_true_densenet, y_pred_densenet)

    # Plot with color
    plt.plot(recall_densenet, precision_densenet, label=f'Class {blood_cell_classes[i]} Densenet PR Curve', color=pal[i])

# Add title and labels
plt.title('Precision-Recall Curves for BloodDenseNet121 - All Classes', fontsize=18)
plt.xlabel('Recall', fontsize=16)
plt.ylabel('Precision', fontsize=16)

# Add legend
plt.legend(loc='best')

# Show plot
plt.show()

The precision-recall curves for the BloodDenseNet121 model show that the Lymphocyte and Monocyte classes maintain high precision above 0.9. The Neutrophil class also shows a strong performance. The Eosinophil curve, however, demonstrates lower precision at similar recall levels, dropping below 0.8 at about 0.6 recall, indicating more misclassifications for this class.

In [ ]:
plot_learning_curves(history_densenet, suptitle="BloodDenseNet121 Training Performance on Blood Cell Dataset")

The plot displays the BloodDenseNet121 model's training and validation performance. Training accuracy consistently increases from 0.35 to over 0.8. Validation accuracy also shows a general upward trend, rising from 0.62 to over 0.85, with a minor dip at epoch 8. Both training and validation losses steadily decrease over the epochs. The training loss drops from approximately 3.1 to 1.6, while the validation loss decreases from 2.5 to 1.48. This consistent improvement in both accuracy and loss indicates the model is learning effectively without significant overfitting.

### BloodInception

The Inception model is a deep convolutional neural network designed to capture multi-scale features efficiently using parallel convolutional layers of different filter sizes. Its architecture reduces computational cost while maintaining high accuracy by combining outputs from multiple kernels. In biomedical imaging, Inception is effective for recognising complex patterns and textures, such as varying cell shapes and structures, improving classification performance without excessive model size.


In [ ]:
# Training inceptionV3
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [ ]:
def create_inception_model(base_model):

    # Flatten thhe output of base model
    x = Flatten()(base_model.output)

    # Add dense layers
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.5)(x)

    # Add output layer
    x = Dense(4, activation='softmax')(x)

    # Return model
    return Model(inputs=base_model.input, outputs=x)

In [ ]:
# Inception Model Training and Evaluation
blood_inception_model = create_inception_model(base_model)
blood_inception_model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Train the model
history_blood_inception = blood_inception_model.fit(train_generator, epochs=10, validation_data=valid_generator)

In [ ]:
# List to store true and predicted labels
true_labels = []
predicted_labels = []

# Iterate over the validation generator
for i in range(validation_steps):
    x_batch, y_batch = valid_generator[i]
    true_labels.extend(np.argmax(y_batch, axis=1))

    # Predict Labels using the Model
    predictions = blood_inception_model.predict(x_batch, verbose=0)
    predicted_labels.extend(np.argmax(predictions, axis=1))

#  Convert to NumPy Arrays
inception_true_labels = np.array(true_labels)
inception_predicted_labels = np.array(predicted_labels)

blood_inception_pred_probs = blood_inception_model.predict(valid_generator, steps=validation_steps, verbose=0)
blood_inception_y_true_onehot = np.zeros_like(blood_inception_pred_probs)
blood_inception_y_true_onehot[np.arange(len(inception_true_labels)), inception_true_labels] = 1

In [ ]:
# Compute Metrics
blood_inception_accuracy = accuracy_score(inception_true_labels, inception_predicted_labels)
blood_inception_precision = precision_score(inception_true_labels, inception_predicted_labels, average='weighted')
blood_inception_recall = recall_score(inception_true_labels, inception_predicted_labels, average='weighted')
blood_inception_f1 = f1_score(inception_true_labels, inception_predicted_labels, average='weighted')
blood_inception_hamming = hamming_loss(inception_true_labels, inception_predicted_labels)
blood_inception_brier_score = np.mean(np.sum((blood_inception_pred_probs - blood_inception_y_true_onehot) ** 2, axis=1))

In [ ]:
print("BloodInception Model Metrics Evaluation Results")
print("-" * 30)
print(f"Accuracy: {blood_inception_accuracy:.4f}")
print(f"Precision: {blood_inception_precision:.4f}")
print(f"Recall: {blood_inception_recall:.4f}")
print(f"F1-score: {blood_inception_f1:.4f}")
print(f"Hamming Loss: {blood_inception_hamming:.4f}")
print(f"Brier Score: {blood_inception_brier_score:.4f}")
print("-" * 30)

The BloodInception model demonstrates excellent performance with an accuracy of 0.9798, indicating almost 98 percent correct predictions. Precision and recall are both 0.9798, showing that positive predictions are highly accurate and most actual positives are detected. The F1-score of 0.9797 confirms a strong balance between precision and recall. Very low Hamming Loss of 0.0202 and Brier Score of 0.0323 indicate minimal classification errors and highly reliable probability estimates. Overall, the model is highly effective for blood cell classification tasks.


In [ ]:
# Classification Report
report = classification_report(inception_true_labels, inception_predicted_labels, target_names=valid_generator.class_indices.keys())
print(report)

In [ ]:
# Generate predictions and true labels if not already done
blood_inception_predictions = blood_inception_model.predict(valid_generator, verbose=1)
blood_inception_predicted_labels = np.argmax(blood_inception_predictions, axis=1)
blood_inception_true_labels = valid_generator.classes

# Calculate the confusion matrix
blood_inception_conf_matrix = confusion_matrix(blood_inception_true_labels, blood_inception_predicted_labels)

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(blood_inception_conf_matrix,annot=True, fmt='d',cmap="Reds", cbar=True, ax=ax, linewidths=1, linecolor="black",  annot_kws={"size": 13}, xticklabels=blood_cell_classes, yticklabels=blood_cell_classes )

# Add title and labels
ax.set_title('Confusion Matrix for BloodInception Model', fontsize=18)
ax.set_xlabel('Predicted Label', fontsize=16)
ax.set_ylabel('True Label', fontsize=16)

# Show plot
plt.show()

The confusion matrix for the BloodInception model shows high accuracy in classifying blood cells. The model correctly identified 484 Eosinophils, 497 Lymphocytes, 495 Monocytes, and 476 Neutrophils. Misclassifications were minimal, with 13 Neutrophils incorrectly classified as Eosinophils. A small number of Eosinophils were also misclassified as Neutrophils 11, and Monocytes 4. Only 1 Monocyte was misclassified as a Lymphocyte, and small numbers of Neutrophils were misclassified as Lymphocytes 5 and Monocytes 6. These low error rates indicate the model performs exceptionally well across all four classes.

In [ ]:
plt.figure(figsize=(8, 6))

# Precision-Recall Curves for inception model
for i in range(len(blood_cell_classes)):
    y_true_inception = (blood_inception_true_labels == i).astype(int)
    y_pred_inception = blood_inception_predictions[:, i]
    precision_inception, recall_inception, _ = precision_recall_curve(y_true_inception, y_pred_inception)

    # Plot with color
    plt.plot(recall_inception, precision_inception, label=f'Class {blood_cell_classes[i]} inception PR Curve', color=pal[i])

# Add title and labels
plt.title('Precision-Recall Curves for Bloodnception - All Classes', fontsize=18)
plt.xlabel('Recall', fontsize=16)
plt.ylabel('Precision', fontsize=16)

# Add legend
plt.legend(loc='best')

# Show plot
plt.show()

The precision-recall curves for the BloodInception model show high performance across all classes. The curves for Lymphocyte, Monocyte, and Neutrophil remain at or near a precision of 1.0 until a very high recall, indicating almost perfect classification. The Eosinophil curve also maintains a very high precision, dropping slightly around 0.95 recall. The model demonstrates strong predictive power with minimal trade-offs.

In [ ]:
plot_learning_curves(history_blood_inception, suptitle="BloodInception Training Performance on Blood Cell Dataset")

The plot shows the BloodInception model's training performance. Training accuracy consistently increases from 0.61 to 0.93. Validation accuracy rises sharply to 0.92 at epoch one, fluctuates, and then increases to over 0.96. Training loss drops from 2.5 to approximately 0.4. Validation loss also decreases, with fluctuations, reaching its lowest point below 0.25. The high accuracy and low loss values, particularly in validation, indicate strong model performance. The slight divergence between training and validation loss suggests a small degree of overfitting, though it remains minimal.

## Model Comparison

In [ ]:
# Collect metrics into a dictionary
results = {
    "Model": ["BloodCNN", "BloodDenseNet121", "BloodInception"],
    "Accuracy": [blood_cnn_accuracy, blood_densenet_accuracy, blood_inception_accuracy],
    "Precision": [blood_cnn_precision, blood_densenet_precision, blood_inception_precision],
    "Recall": [blood_cnn_recall, blood_densenet_recall, blood_inception_recall],
    "F1-score": [blood_cnn_f1, blood_densenet_f1, blood_inception_f1],
    "Hamming Loss": [blood_cnn_hamming, blood_densenet_hamming, blood_inception_hamming],
    "Brier Score": [blood_cnn_brier_score, blood_densenet_brier_score, blood_inception_brier_score]
}

# Create DataFrame
metrics_df = pd.DataFrame(results)


# Save to CSV
metrics_df.to_csv("blood_models_metrics.csv", index=False)
print("Metrics saved to blood_models_metrics.csv ✅")

In [ ]:
metrics_df.head()

In [ ]:
# Prepare data for plotting
models = ['BloodCnn', 'BloodDenseNet121', 'BloodnInception']
accuracies = [blood_cnn_accuracy, blood_densenet_accuracy, blood_inception_accuracy]

# Create the bar chart
plt.figure(figsize=(8, 6))
bars = plt.bar(models, accuracies, color=pal, edgecolor='black', linewidth=1.5)


# Add titles and labels
plt.title('Blood Cell Classification Model Accuracy Comparison', fontsize=18)
plt.xlabel('Model', fontsize=16)
plt.ylabel('Accuracy', fontsize=16)
plt.ylim([0, 1.1])

# Add ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Annotate
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 0.01,
             f'{yval:.4f}', ha='center', va='bottom', fontsize=14)

# Show plot
plt.show()

The bar chart compares the accuracy of three blood cell classification models. The BloodCnn model achieved an accuracy of 0.9032. The BloodDenseNet121 model had the lowest performance with an accuracy of 0.8412. The BloodInception model was the most accurate, reaching a high of 0.9798. This demonstrates that the BloodInception model is the superior performer.

In [ ]:
# hamming loss
hamming_losses = [blood_cnn_hamming, blood_densenet_hamming, blood_inception_hamming]

# Horizontal Bar Plot for Hamming Loss Comparison
plt.figure(figsize=(8, 6))

# Plotting the horizontal bars
bars = plt.barh(models, hamming_losses, color=pal, edgecolor='black', linewidth=1.5)

# Add title and labels
plt.title('Model Hamming Loss Comparison', fontsize=18)
plt.xlabel('Hamming Loss', fontsize=16)
plt.ylabel('Model', fontsize=16)

# Add limit
plt.xlim([0, 1])

# Add ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Annotate
for bar in bars:
    plt.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():.4f}', va='center', fontsize=16, color='black')

# Show plot
plt.show()

The chart compares the Hamming loss of three models. BloodCnn has a Hamming loss of 0.0968, while BloodDenseNet121 has a significantly higher loss of 0.1588. The BloodInception model has the lowest Hamming loss at 0.0202, indicating it is the most accurate at predicting the correct class without making additional errors.

In [ ]:
# Metrics values
precisions = [blood_cnn_precision, blood_densenet_precision, blood_inception_precision]
recalls = [blood_cnn_recall, blood_densenet_recall, blood_inception_recall]
f1_scores = [blood_cnn_f1, blood_densenet_f1, blood_inception_f1]

# Plot subplots
fig, axs = plt.subplots(1, 3, figsize=(20, 6))

# Precision plot
axs[0].plot(models, precisions, marker='o', color=pal[0], linewidth=2)

# Add title and labels
axs[0].set_title("Precision", fontsize=18)
axs[0].set_ylabel("Score", fontsize=16)

# Add ticks
axs[0].tick_params(axis='x', labelsize=15)
axs[0].tick_params(axis='y', labelsize=15)

# Recall plot
axs[1].plot(models, recalls, marker='o', color=pal[1], linewidth=2)

# Add title and labels
axs[1].set_title("Recall", fontsize=18)
axs[1].set_ylabel("Score", fontsize=16)

# Add ticks
axs[1].tick_params(axis='x', labelsize=15)
axs[1].tick_params(axis='y', labelsize=15)

# F1-score plot
axs[2].plot(models, f1_scores, marker='o', color=pal[2], linewidth=2)

# Add title and labels
axs[2].set_title("F1-Score", fontsize=18)
axs[2].set_ylabel("Score", fontsize=16)

# Add ticks
axs[2].tick_params(axis='x', labelsize=15)
axs[2].tick_params(axis='y', labelsize=15)

# Add main title
plt.suptitle("Model Evaluation Metrics Comparison", fontsize=25)

# Show plot
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

The plot shows the performance of three models across precision, recall, and F1-score. For all three metrics, BloodDenseNet121 has the lowest score: 0.844 for precision, 0.842 for recall, and 0.844 for F1-score. BloodCnn performs better, with a precision of 0.903, recall of 0.903, and F1-score of 0.903. The BloodInception model is the clear leader, achieving the highest scores: a precision of 0.979, recall of 0.979, and F1-score of 0.979. These results consistently show BloodInception as the best-performing model.

In [ ]:
# Brier Scores
brier_scores = [blood_cnn_brier_score, blood_densenet_brier_score, blood_inception_brier_score]

# Dot plot
plt.figure(figsize=(8, 6))
plt.scatter(models, brier_scores, color=pal[:len(models)], s=200, edgecolors="black")

# Add title and labels
plt.title("Brier Score Comparison Across Models", fontsize=20)
plt.ylabel("Brier Score", fontsize=16)

# Add y limit
plt.ylim(0.005, 0.270)

# Add ticks
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Annotations
for i, score in enumerate(brier_scores):
    plt.text(models[i], score + 0.002, f"{score:.4f}", ha='center', fontsize=12, color="black")

# Show plot
plt.show()

The plot compares the Brier scores of three models. The BloodCnn model has a Brier score of 0.1448. The BloodDenseNet121 model has the highest Brier score at 0.2256, indicating the least accurate probability predictions. The BloodInception model has the lowest score of 0.0323, showing it is the best-calibrated model with the most reliable probability estimates.

## Model Testing

In [ ]:
# Saving the best model
blood_densenet_model.save('model.h5')
print("BloodDenseNet Model saved as 'model.h5'")

In [ ]:
from tensorflow.keras.models import load_model

# Load the hybrid model
best_model = load_model('model.h5')
print("Model model loaded successfully.")

In [ ]:
# Predict using the model
test_predictions = best_model.predict(test_generator, verbose=1)

# Get predicted labels
predicted_labels = np.argmax(test_predictions, axis=1)
true_labels = test_generator.classes  # True labels from the generator

In [ ]:
# Confusion Matrix with increased font size
conf_matrix = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(10, 7))

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(conf_matrix,annot=True, fmt='d',cmap="Reds", cbar=True, ax=ax, linewidths=1, linecolor="black",  annot_kws={"size": 13}, xticklabels=blood_cell_classes, yticklabels=blood_cell_classes )

# Add title and labels
ax.set_title('Confusion Matrix for Best Model - Test Data', fontsize=18)
ax.set_xlabel('Predicted Label', fontsize=16)
ax.set_ylabel('True Label', fontsize=16)

# Show plot
plt.show()

The confusion matrix shows the performance of the best model on the test data. The model correctly classified 9 Eosinophils, 6 Lymphocytes, 4 Monocytes, and 35 Neutrophils. However, it struggled with Eosinophil and Neutrophil classification. A notable number of Neutrophils 9 were misclassified as Eosinophils. Also, 2 Neutrophils were misclassified as Lymphocytes and 2 as Monocytes. The model misclassified 2 Eosinophils as Lymphocytes, and 1 as Monocyte, and 1 as Neutrophil. This indicates strong performance for Lymphocytes and Monocytes, but a specific challenge in distinguishing between Eosinophils and Neutrophils.

In [ ]:
# Predict using the model
test_predictions = best_model.predict(test_generator, verbose=1)

# Get predicted labels and true labels
predicted_labels = np.argmax(test_predictions, axis=1)
true_labels = test_generator.classes  # True labels from the generator

# Calculate accuracy
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Accuracy: {accuracy:.4f}")

# Map class indices to their labels
class_indices = test_generator.class_indices
index_to_class = {v: k for k, v in class_indices.items()}

# Get file names
file_names = test_generator.filenames

# Display predicted and actual classes along with file names
print("\nPredicted and Actual Classes with File Names:")
for i, (file_name, predicted, actual) in enumerate(zip(file_names, predicted_labels, true_labels)):
    predicted_class = index_to_class[predicted]
    actual_class = index_to_class[actual]
    print(f"Sample {i+1}: File - {file_name}, Predicted - {predicted_class}, Actual - {actual_class}")


In [ ]:
import json

# Save class indices to a JSON file
class_indices = test_generator.class_indices
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f)

print("Class indices saved to 'class_indices.json'")

## Failed Attempts

Several experimental attempts failed to achieve satisfactory results in blood cell classification. Models such as VGG with squeeze-and-excitation blocks and EfficientNetB0 with squeeze-and-excitation blocks struggled to cross 0.4 accuracy. A simple baseline model also performed poorly, highlighting its inability to capture complex features. Additionally, attention and multi-head attention mechanisms did not yield significant improvements. These results suggest that not all advanced architectures or attention strategies are effective for this specific dataset.


## Conclusion

The study applied deep learning methods including CNN, fine-tuned DenseNet121, and Inception for blood cell classification. Among the models, BloodInception achieved the best performance with an accuracy of 0.9798, showing its effectiveness in capturing complex cell features. BloodCNN also performed well with an accuracy of 0.9032, indicating reliable generalisation. In contrast, BloodDenseNet121 recorded the lowest accuracy of 0.8412, making it the weakest among the tested models. Overall, Inception proved to be the most suitable method for this task.
